
# Mission C - Collecte des prix du Gazole avec Python

## Projet : OptiFleet Logistics

### Objectif

Interroger l'API publique des prix des carburants pour récupérer toutes les stations-service du département 69,
y compris celles dont le prix du Gazole est manquant.

Les contrôles qualité et les filtres métier seront appliqués ensuite avec Pandas.

### Étapes du notebook

1. Importer la bibliothèque requests.
2. Envoyer une requête HTTP GET.
3. Vérifier la réponse de l'API.
4. Lire les données JSON.
5. Transformer les résultats en DataFrame Pandas.
6. Récupérer toutes les stations avec la pagination.
7. Effectuer les premiers contrôles de qualité.

### Source

API Explore v2.1 - Prix des carburants en France,
Flux instantané v2.

### Périmètre initial

Département du Rhône (code 69).

Le périmètre final du projet sera la Métropole de Lyon.


In [2]:
# Vérifier la bibliothèque requests
import json
from datetime import datetime, timezone
from pathlib import Path
import requests

print("Bibliothèque requests disponible !")
print("Version :", requests.__version__)

Bibliothèque requests disponible !
Version : 2.32.5


In [3]:
URL = (
    "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/"
    "prix-des-carburants-en-france-flux-instantane-v2/records"
)

params = {
    "where": 'code_departement="69"',
    "limit": 5
}

reponse = requests.get(URL, params=params, timeout=30)

print("Code HTTP :", reponse.status_code)
print("URL appelée :", reponse.url)

Code HTTP : 200
URL appelée : https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-des-carburants-en-france-flux-instantane-v2/records?where=code_departement%3D%2269%22&limit=5


In [4]:
reponse.raise_for_status()

donnees = reponse.json()

print("Type :", type(donnees))
print("Clés :", donnees.keys())
print("Total :", donnees["total_count"])
print("Stations reçues :", len(donnees["results"]))

Type : <class 'dict'>
Clés : dict_keys(['total_count', 'results'])
Total : 200
Stations reçues : 5


## C2 - Pagination automatique des stations-service

Objectif : récupérer toutes les stations du département 69,
y compris celles sans prix Gazole renseigné.

L'API autorise au maximum 100 résultats par page.

Une boucle while permettra de récupérer automatiquement
toutes les pages, quel que soit le nombre de stations.

In [5]:
# Paramètres de départ
taille_page = 100
offset = 0

toutes_stations = []
total_annonce = None

while True:

    # Construire les paramètres de la page courante
    params_page = {
        "where": 'code_departement="69"',
        "order_by": "id ASC",
        "limit": taille_page,
        "offset": offset
    }

    # Envoyer la requête
    reponse_page = requests.get(
        URL,
        params=params_page,
        timeout=30
    )

    # Vérifier la réponse HTTP
    reponse_page.raise_for_status()

    # Convertir le JSON
    donnees_page = reponse_page.json()

    # Lire le total annoncé lors de la première requête
    if total_annonce is None:
        total_annonce = donnees_page["total_count"]

    # Récupérer les stations de cette page
    stations_page = donnees_page["results"]

    # Ajouter les stations à notre liste générale
    toutes_stations.extend(stations_page)

    print(
        f"Offset {offset} : "
        f"{len(stations_page)} stations reçues"
    )

    # Arrêter si la page est vide ou si le total est atteint
    if not stations_page or len(toutes_stations) >= total_annonce:
        break

    # Passer à la page suivante
    offset += taille_page

print("Total annoncé :", total_annonce)
print("Total collecté :", len(toutes_stations))

Offset 0 : 100 stations reçues
Offset 100 : 100 stations reçues
Total annoncé : 200
Total collecté : 200


In [6]:
ids = [station["id"] for station in toutes_stations]

print("Total annoncé :", total_annonce)
print("Stations collectées :", len(toutes_stations))
print("Identifiants uniques :", len(set(ids)))
print("Doublons :", len(ids) - len(set(ids)))

assert len(toutes_stations) == total_annonce, (
    "Collecte incomplète : nombre de stations différent du total annoncé."
)

assert len(ids) == len(set(ids)), (
    "Des identifiants sont présents plusieurs fois."
)

Total annoncé : 200
Stations collectées : 200
Identifiants uniques : 200
Doublons : 0


In [7]:
print("Dossier de travail :", Path.cwd())

Dossier de travail : C:\Users\bowma\optifleet-logistics\notebooks


In [8]:
dossier_raw = Path("../data/raw")
dossier_raw.mkdir(parents=True, exist_ok=True)

print("Dossier de sauvegarde :", dossier_raw.resolve())

Dossier de sauvegarde : C:\Users\bowma\optifleet-logistics\data\raw


In [9]:
# Enregistrer l'heure actuelle en UTC
date_collecte = datetime.now(timezone.utc)

# Construire le dictionnaire à sauvegarder
collecte_brute = {
    "date_collecte": date_collecte.isoformat(),
    "source": URL,
    "filtre": params_page["where"],
    "total_count": total_annonce,
    "nombre_collecte": len(toutes_stations),
    "results": toutes_stations
}

In [10]:
# Créer un nom de fichier unique avec la date et l'heure UTC
nom_fichier = (
    f"carburants_69_{date_collecte.strftime('%Y-%m-%d_%H%M%S')}.json"
)

# Construire le chemin complet
chemin_fichier = dossier_raw / nom_fichier

# Écrire le dictionnaire dans un fichier JSON
with open(chemin_fichier, "w", encoding="utf-8") as fichier:
    json.dump(collecte_brute, fichier, ensure_ascii=False, indent=2)

print("Fichier enregistré :", chemin_fichier.resolve())

Fichier enregistré : C:\Users\bowma\optifleet-logistics\data\raw\carburants_69_2026-10-15_080144.json


In [11]:
with open(chemin_fichier, "r", encoding="utf-8") as fichier:
    verification = json.load(fichier)

print("Date de collecte :", verification["date_collecte"])
print("Stations enregistrées :", len(verification["results"]))
print("Fichier existe :", chemin_fichier.exists())
print("Taille du fichier :", chemin_fichier.stat().st_size, "octets")

Date de collecte : 2026-10-15T08:01:44.944796+00:00
Stations enregistrées : 200
Fichier existe : True
Taille du fichier : 682093 octets


In [12]:
# Récupérer les identifiants des stations sauvegardées
ids = [station["id"] for station in verification["results"]]

# Comparer les quantités
print("Total annoncé :", verification["total_count"])
print("Stations sauvegardées :", len(ids))
print("Stations uniques :", len(set(ids)))
print("Doublons :", len(ids) - len(set(ids)))

Total annoncé : 200
Stations sauvegardées : 200
Stations uniques : 200
Doublons : 0


## C3 - Construction du DataFrame Pandas

Objectif : transformer les données JSON brutes
en tableau Pandas pour faciliter leur analyse.

Source : fichier JSON sauvegardé dans data/raw/.

In [13]:
import pandas as pd

# Transformer la liste de dictionnaires en DataFrame
df_brut = pd.DataFrame(verification["results"])

# Afficher les dimensions du tableau
print("Dimensions :", df_brut.shape)

# Afficher les cinq premières lignes
display(df_brut.head())

Dimensions : (200, 47)


,id,latitude,longitude,cp,pop,adresse,ville,horaires,services,prix,...,carburants_indisponibles,carburants_rupture_temporaire,carburants_rupture_definitive,horaires_automate_24_24,services_service,departement,code_departement,region,code_region,horaires_jour
0,69002006,4573500,481800,69002,R,84 QUAI PERRACHE,Lyon,"{""@automate-24-24"": """", ""jour"": [{""@id"": ""1"", ...","{""service"": [""Boutique alimentaire"", ""Boutique...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, E85, GPLc]",None,None,Non,"[Boutique alimentaire, Boutique non alimentair...",Rhône,69,Auvergne-Rhône-Alpes,84,"Lundi00.00-23.59, Mardi00.00-23.59, Mercredi00..."
1,69003012,4574700,486700,69003,R,65 COURS ALBERT THOMAS,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Boutique alimentaire"", ""Boutique...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, GPLc, SP98]",None,E85;GPLc;SP98,Oui,"[Boutique alimentaire, Boutique non alimentair...",Rhône,69,Auvergne-Rhône-Alpes,84,"Automate-24-24, Lundi06.00-21.00, Mardi06.00-2..."
2,69003013,4575446.02,485283.65475131,69003,R,258 rue Garibaldi,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Toilettes publiques"", ""Boutique ...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, SP98]",None,E85;SP98,Oui,"[Toilettes publiques, Boutique alimentaire, Bo...",Rhône,69,Auvergne-Rhône-Alpes,84,"Automate-24-24, Lundi06.00-22.00, Mardi06.00-2..."
3,69004002,4577600,482100,69004,R,22 Rue Philippe de Lassalle,Lyon,"{""@automate-24-24"": """", ""jour"": [{""@id"": ""1"", ...",None,"[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, GPLc]",None,None,Non,None,Rhône,69,Auvergne-Rhône-Alpes,84,None
4,69004016,4577900,482800,69004,R,97 RUE DENFERT ROCHEREAU,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Toilettes publiques"", ""Boutique ...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, E85, GPLc]",SP95;E85;GPLc,None,Oui,"[Toilettes publiques, Boutique alimentaire, St...",Rhône,69,Auvergne-Rhône-Alpes,84,Automate-24-24


In [14]:
# Compter toutes les stations
nb_stations = len(df_brut)

# Compter les stations sans prix Gazole
nb_prix_manquants = df_brut["gazole_prix"].isna().sum()

# Calculer le pourcentage de prix manquants
pourcentage_manquants = (
    nb_prix_manquants / nb_stations
) * 100

print("Stations collectées :", nb_stations)
print("Prix Gazole manquants :", nb_prix_manquants)
print(f"Pourcentage manquant : {pourcentage_manquants:.2f} %")

Stations collectées : 200
Prix Gazole manquants : 8
Pourcentage manquant : 4.00 %


### C3.2 - Sélection des colonnes utiles

Objectif : créer un DataFrame de travail contenant
uniquement les colonnes nécessaires à l'analyse
des prix du Gazole pour OptiFleet Logistics.

Le DataFrame brut sera conservé sans modification.

In [15]:
# Définir les colonnes nécessaires à l'analyse
colonnes_utiles = [
    "id",
    "adresse",
    "ville",
    "cp",
    "geom",
    "gazole_prix",
    "gazole_maj",
    "code_departement"
]

# Créer un nouveau DataFrame à partir des colonnes sélectionnées
df = df_brut[colonnes_utiles].copy()

# Vérifier les dimensions
print("Dimensions du DataFrame brut :", df_brut.shape)
print("Dimensions du DataFrame de travail :", df.shape)

# Afficher les cinq premières stations
display(df.head())

Dimensions du DataFrame brut : (200, 47)
Dimensions du DataFrame de travail : (200, 8)


,id,adresse,ville,cp,geom,gazole_prix,gazole_maj,code_departement
0,69002006,84 QUAI PERRACHE,Lyon,69002,"{'lon': 4.818, 'lat': 45.735}",2.489,2026-10-14T15:21:57+00:00,69
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,"{'lon': 4.867, 'lat': 45.747}",2.429,2026-10-14T20:50:00+00:00,69
2,69003013,258 rue Garibaldi,Lyon,69003,"{'lon': 4.8528365475131, 'lat': 45.7544602}",2.449,2026-10-05T21:45:00+00:00,69
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,"{'lon': 4.821, 'lat': 45.776}",2.343,2026-10-15T07:32:00+00:00,69
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,"{'lon': 4.828, 'lat': 45.779}",2.469,2026-10-14T21:00:00+00:00,69


In [16]:
# Vérifier que les prix manquants ont été conservés
print("Stations dans df :", len(df))
print("Prix Gazole manquants :", df["gazole_prix"].isna().sum())

# Afficher les stations dont le prix est absent
display(
    df.loc[
        df["gazole_prix"].isna(),
        ["id", "ville", "adresse", "gazole_prix"]
    ]
)

Stations dans df : 200
Prix Gazole manquants : 8


,id,ville,adresse,gazole_prix
18,69100022,Bron,283 RTE DE GENAS,NaN
53,69200009,Vénissieux,RTE DE CORBAS - RD 57,NaN
72,69250009,Neuville-sur-Saône,28 AVENUE BURDEAU,NaN
109,69380014,Lozanne,370 ROUTE DE LA TOUR DE SALVAGNY,NaN
121,69400016,Arnas,2315 ROUTE DE LONGSARD,NaN
173,69700002,Loire-sur-Rhône,1293 Route de Beaucaire (Ex.45 R.N. 86),NaN
194,69870001,Lamure-sur-Azergues,890 rue centrale,NaN
197,69930001,Saint-Laurent-de-Chamousset,La Croix Blanche,NaN


### C3.3 - Gestion des dates et fraîcheur des prix

Objectif : calculer l'ancienneté de chaque prix du Gazole
par rapport à la date de collecte du fichier JSON.

Les calculs sont effectués en UTC pour éviter
les erreurs liées aux fuseaux horaires.

In [17]:
# 1. Lire l'horodatage de collecte depuis le JSON
horodatage_collecte = pd.to_datetime(
    verification["date_collecte"],
    utc=True
)

# 2. Ajouter l'horodatage UTC à toutes les stations
df["horodatage_collecte"] = horodatage_collecte

# 3. Convertir en heure de Paris pour obtenir la date métier
df["date_collecte"] = (
    df["horodatage_collecte"]
    .dt.tz_convert("Europe/Paris")
    .dt.date
)

# 4. Vérifier les résultats
display(
    df[
        ["id", "ville", "horodatage_collecte", "date_collecte"]
    ].head())

,id,ville,horodatage_collecte,date_collecte
0,69002006,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
1,69003012,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
2,69003013,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
3,69004002,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
4,69004016,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15


In [18]:
# 1. Convertir gazole_maj en datetime avec fuseau UTC
df["gazole_maj"] = pd.to_datetime(
    df["gazole_maj"],
    errors="coerce",
    utc=True
)

# 2. Calculer la différence entre la collecte et la mise à jour
difference = (
    df["horodatage_collecte"] - df["gazole_maj"]
)

# 3. Convertir cette différence en heures
df["age_prix_heures"] = (
    difference.dt.total_seconds() / 3600
)

# 4. Vérifier les types
print("Type gazole_maj :", df["gazole_maj"].dtype)
print("Type horodatage_collecte :", df["horodatage_collecte"].dtype)

# 5. Afficher les résultats
display(
    df[
        [
            "id",
            "ville",
            "gazole_prix",
            "gazole_maj",
            "horodatage_collecte",
            "age_prix_heures"
        ]
    ].head()
)

Type gazole_maj : datetime64[ns, UTC]
Type horodatage_collecte : datetime64[ns, UTC]


,id,ville,gazole_prix,gazole_maj,horodatage_collecte,age_prix_heures
0,69002006,Lyon,2.489,2026-10-14 15:21:57+00:00,2026-10-15 08:01:44.944796+00:00,16.663318
1,69003012,Lyon,2.429,2026-10-14 20:50:00+00:00,2026-10-15 08:01:44.944796+00:00,11.195818
2,69003013,Lyon,2.449,2026-10-05 21:45:00+00:00,2026-10-15 08:01:44.944796+00:00,226.279151
3,69004002,Lyon,2.343,2026-10-15 07:32:00+00:00,2026-10-15 08:01:44.944796+00:00,0.495818
4,69004016,Lyon,2.469,2026-10-14 21:00:00+00:00,2026-10-15 08:01:44.944796+00:00,11.029151


In [19]:
# Identifier les dates de mise à jour postérieures à la collecte
prix_futurs = df[df["age_prix_heures"] < 0]

print("Nombre de prix datés après la collecte :", len(prix_futurs))

display(
    prix_futurs[
        ["id", "ville", "gazole_maj",
         "date_collecte", "age_prix_heures"]
    ]
)

Nombre de prix datés après la collecte : 10


,id,ville,gazole_maj,date_collecte,age_prix_heures
33,69130005,Écully,2026-10-15 08:14:35+00:00,2026-10-15,-0.213904
34,69132001,Écully,2026-10-15 08:26:00+00:00,2026-10-15,-0.404182
117,69400008,Gleizé,2026-10-15 09:27:00+00:00,2026-10-15,-1.420849
120,69400015,Gleizé,2026-10-15 08:53:27+00:00,2026-10-15,-0.861682
152,69550002,Amplepuis,2026-10-15 08:48:21+00:00,2026-10-15,-0.776682
168,69658001,Villefranche-sur-Saône,2026-10-15 08:17:24+00:00,2026-10-15,-0.260849
182,69800010,Saint-Priest,2026-10-15 09:07:24+00:00,2026-10-15,-1.094182
185,69800014,Saint-Priest,2026-10-15 09:00:00+00:00,2026-10-15,-0.970849
192,69830003,Saint-Georges-de-Reneins,2026-10-15 08:37:26+00:00,2026-10-15,-0.594738
199,69970001,Chaponnay,2026-10-15 09:23:57+00:00,2026-10-15,-1.370015


In [20]:
# Vérifier l'ampleur des écarts temporels négatifs

print(
    "Âge minimum observé (h) :",
    df["age_prix_heures"].min()
)

print(
    "Dates futures de plus de 2 heures :",
    (df["age_prix_heures"] < -2).sum()
)

Âge minimum observé (h) : -1.4208486677777779
Dates futures de plus de 2 heures : 0


In [21]:
# Choisir une station présentant une date future apparente
id_station = 69400008

# Retrouver sa ligne dans les données brutes
station_brute = df_brut.loc[
    df_brut["id"] == id_station
]

# Afficher les informations d'origine
print("Station :", id_station)
print("Date gazole_maj :", station_brute["gazole_maj"].iloc[0])
print("Champ prix brut :")
print(station_brute["prix"].iloc[0])

Station : 69400008
Date gazole_maj : 2026-10-15T09:27:00+00:00
Champ prix brut :
[{"@nom": "Gazole", "@id": "1", "@maj": "2026-10-15 09:27:00", "@valeur": "2.399"}, {"@nom": "E85", "@id": "3", "@maj": "2026-10-15 09:27:00", "@valeur": "0.869"}, {"@nom": "E10", "@id": "5", "@maj": "2026-10-15 09:27:00", "@valeur": "2.209"}, {"@nom": "SP98", "@id": "6", "@maj": "2026-10-15 09:27:00", "@valeur": "2.300"}]


### Conclusion provisoire - Anomalie temporelle

Lors de la collecte du 15 octobre 2026, 10 stations sur 200
présentent une date de mise à jour apparemment postérieure
à la collecte.

Pour la station 69400008, le champ brut `prix` indique
`2026-10-15 09:27:00` sans fuseau horaire, tandis que
`gazole_maj` indique `2026-10-15T09:27:00+00:00`.

Cette observation est compatible avec l'hypothèse d'une
heure locale interprétée comme UTC, mais ne suffit pas
à confirmer une erreur de fuseau horaire.

**Décision :** conserver les horodatages d'origine,
signaler les dates futures apparentes et ne pas appliquer
de correction automatique sans validation supplémentaire.

### C3.4 - Classification de la fraîcheur des prix

Objectif : identifier les prix suffisamment récents
pour être utilisés dans les recommandations OptiFleet.

Règle métier :
- Âge compris entre 0 et 48 heures : récent.
- Âge supérieur à 48 heures : ancien.
- Âge négatif : anomalie temporelle.
- Date manquante : fraîcheur inconnue.

In [22]:
# 1. Un prix est récent si son âge est compris entre 0 et 48 h
condition_recent = df["age_prix_heures"].between(0, 48)

# 2. Le prix doit également être renseigné
condition_prix_present = df["gazole_prix"].notna()

# 3. Créer l'indicateur final
df["prix_recent_48h"] = (
    condition_recent & condition_prix_present
)

# 4. Compter les stations éligibles
print(df["prix_recent_48h"].value_counts(dropna=False))

prix_recent_48h
True     121
False     79
Name: count, dtype: int64


In [24]:
# 1. Compter les prix récents et renseignés
nb_recents = df["prix_recent_48h"].sum()

# 2. Compter les prix anciens et renseignés
nb_anciens = (
    (df["age_prix_heures"] > 48)
    & df["gazole_prix"].notna()
).sum()

# 3. Compter les dates futures apparentes
nb_futurs = (
    (df["age_prix_heures"] < 0)
    & df["gazole_prix"].notna()
).sum()

# 4. Compter les prix Gazole manquants
nb_prix_manquants = df["gazole_prix"].isna().sum()

# 5. Compter les dates de mise à jour manquantes
# uniquement pour les stations avec un prix
nb_dates_manquantes = (
    df["age_prix_heures"].isna()
    & df["gazole_prix"].notna()
).sum()

# 6. Afficher les résultats
print("Prix récents (0 - 48 h) :", nb_recents)
print("Prix anciens (> 48 h) :", nb_anciens)
print("Dates futures apparentes :", nb_futurs)
print("Prix Gazole manquants :", nb_prix_manquants)
print("Dates manquantes avec prix :", nb_dates_manquantes)

# 7. Vérifier que chaque station appartient
# à une seule catégorie
total_controle = (
    nb_recents
    + nb_anciens
    + nb_futurs
    + nb_prix_manquants
    + nb_dates_manquantes
)

print("-----------------------------")
print("Total contrôlé :", total_controle)
print("Total stations :", len(df))
print("Contrôle réussi :", total_controle == len(df))

Prix récents (0 - 48 h) : 121
Prix anciens (> 48 h) : 61
Dates futures apparentes : 10
Prix Gazole manquants : 8
Dates manquantes avec prix : 0
-----------------------------
Total contrôlé : 200
Total stations : 200
Contrôle réussi : True


## C4 - Contrôles qualité et préparation des données

### C4.1 - Contrôle des prix du Gazole

Objectif : vérifier que les prix sont renseignés,
numériques et situés dans une plage plausible.

Règle provisoire : entre 0,50 et 4,00 €/L.

Aucune ligne ne sera supprimée avant
l'analyse des anomalies.

In [25]:
# Vérifier le type de données de la colonne
print("Type initial :", df["gazole_prix"].dtype)

# Compter les valeurs manquantes
print("Prix manquants :", df["gazole_prix"].isna().sum())

# Afficher les statistiques descriptives
display(df["gazole_prix"].describe())

Type initial : float64
Prix manquants : 8


count    192.000000
mean       2.363677
std        0.088978
min        2.240000
25%        2.261250
50%        2.359000
75%        2.426750
max        2.649000
Name: gazole_prix, dtype: float64

In [26]:
df["gazole_prix"].between(0.50, 4.00)

0       True
1       True
2       True
3       True
4       True
       ...  
195     True
196     True
197    False
198     True
199     True
Name: gazole_prix, Length: 200, dtype: bool

### C4.2 - Détection des prix atypiques avec l'IQR

Objectif : identifier les prix potentiellement atypiques
par rapport à la distribution observée.

Méthode : quartiles Q1 et Q3, écart interquartile
et règle des 1,5 × IQR.

Les prix signalés seront examinés avant
toute décision de correction ou d'exclusion.

In [27]:
# Calculer les quartiles
Q1 = df["gazole_prix"].quantile(0.25)
Q3 = df["gazole_prix"].quantile(0.75)

# Calculer l'écart interquartile
IQR = Q3 - Q1

# Définir les bornes de détection
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

# Afficher les résultats
print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)
print("Borne inférieure :", borne_inf)
print("Borne supérieure :", borne_sup)

Q1 : 2.26125
Q3 : 2.42675
IQR : 0.1655000000000002
Borne inférieure : 2.013
Borne supérieure : 2.6750000000000007


In [28]:
# Créer une condition de détection des prix atypiques
condition_atypique = (
    (df["gazole_prix"] < borne_inf) |
    (df["gazole_prix"] > borne_sup)
)

# Ajouter un indicateur booléen
df["prix_atypique_iqr"] = condition_atypique

# Compter les prix atypiques
print("Prix atypiques :", df["prix_atypique_iqr"].sum())

# Afficher les stations concernées
display(
    df.loc[
        df["prix_atypique_iqr"],
        ["id", "ville", "gazole_prix", "gazole_maj"]
    ]
)

Prix atypiques : 0


,id,ville,gazole_prix,gazole_maj


### C4.3 - Synthèse des contrôles qualité

Objectif : regrouper les résultats des contrôles
dans un tableau de synthèse reproductible.

Contrôles réalisés :
- Unicité des identifiants.
- Présence et plausibilité des prix.
- Détection des valeurs atypiques (IQR).
- Fraîcheur et cohérence des dates.

In [29]:
# Calculer les indicateurs de qualité
controles = {
    "Stations collectées": len(df),
    "Identifiants uniques": df["id"].nunique(),
    "Doublons d'identifiants": df["id"].duplicated().sum(),

    "Prix Gazole manquants": nb_prix_manquants,

    "Prix hors bornes": (
        df["gazole_prix"].notna()
        & ~df["gazole_prix"].between(0.50, 4.00)
    ).sum(),

    "Prix atypiques IQR": df["prix_atypique_iqr"].sum(),

    "Prix récents (0-48 h)": nb_recents,
    "Prix anciens (>48 h)": nb_anciens,
    "Dates futures apparentes": nb_futurs,
    "Dates manquantes avec prix": nb_dates_manquantes
}

# Présenter les contrôles sous forme de tableau
df_controles = pd.DataFrame(
    list(controles.items()),
    columns=["Contrôle", "Nombre"]
)

display(df_controles)

,Contrôle,Nombre
0,Stations collectées,200
1,Identifiants uniques,200
2,Doublons d'identifiants,0
3,Prix Gazole manquants,8
4,Prix hors bornes,0
5,Prix atypiques IQR,0
6,Prix récents (0-48 h),121
7,Prix anciens (>48 h),61
8,Dates futures apparentes,10
9,Dates manquantes avec prix,0


### C4.4 - Préparation des données pour l'export CSV

Objectif : préparer un fichier tabulaire facilement
importable dans BigQuery.

Transformations :
- Extraction de la latitude et de la longitude depuis geom.
- Conservation des indicateurs de qualité.
- Conservation des dates en UTC.
- Export des données au format CSV.

In [30]:
# Créer une copie pour préparer l'export
df_export = df.copy()

# Extraire la latitude et la longitude
df_export["latitude"] = df_export["geom"].apply(
    lambda position: position.get("lat")
    if isinstance(position, dict) else None
)

df_export["longitude"] = df_export["geom"].apply(
    lambda position: position.get("lon")
    if isinstance(position, dict) else None
)

# Vérifier les coordonnées
display(
    df_export[
        ["id", "ville", "latitude", "longitude"]
    ].head()
)

,id,ville,latitude,longitude
0,69002006,Lyon,45.73500,4.818000
1,69003012,Lyon,45.74700,4.867000
2,69003013,Lyon,45.75446,4.852837
3,69004002,Lyon,45.77600,4.821000
4,69004016,Lyon,45.77900,4.828000


In [31]:
print("Latitudes manquantes :", df_export["latitude"].isna().sum())
print("Longitudes manquantes :", df_export["longitude"].isna().sum())

print("Type latitude :", df_export["latitude"].dtype)
print("Type longitude :", df_export["longitude"].dtype)

Latitudes manquantes : 0
Longitudes manquantes : 0
Type latitude : float64
Type longitude : float64


### C4.5  Contrôle de validité des coordonnées

Objectif : vérifier que les coordonnées géographiques
respectent les bornes mondiales de latitude et longitude.

- Latitude : entre -90 et 90.
- Longitude : entre -180 et 180.

Ce contrôle ne garantit pas encore que les stations
se trouvent dans la Métropole de Lyon.

In [32]:
# Vérifier que les coordonnées respectent les bornes géographiques
coordonnees_valides = (
    df_export["latitude"].between(-90, 90)
    & df_export["longitude"].between(-180, 180)
)

# Compter les stations valides et invalides
print("Coordonnées valides :", coordonnees_valides.sum())
print("Coordonnées invalides :", (~coordonnees_valides).sum())

# Afficher les éventuelles stations problématiques
display(
    df_export.loc[
        ~coordonnees_valides,
        ["id", "ville", "latitude", "longitude"]
    ]
)

Coordonnées valides : 200
Coordonnées invalides : 0


,id,ville,latitude,longitude


In [33]:
# Retirer la colonne geom du DataFrame d'export
df_export = df_export.drop(columns=["geom"])

# Vérifier les dimensions et les colonnes
print("Dimensions :", df_export.shape)
print("Colonnes :", df_export.columns.tolist())

display(df_export.head())

Dimensions : (200, 14)
Colonnes : ['id', 'adresse', 'ville', 'cp', 'gazole_prix', 'gazole_maj', 'code_departement', 'horodatage_collecte', 'date_collecte', 'age_prix_heures', 'prix_recent_48h', 'prix_atypique_iqr', 'latitude', 'longitude']


,id,adresse,ville,cp,gazole_prix,gazole_maj,code_departement,horodatage_collecte,date_collecte,age_prix_heures,prix_recent_48h,prix_atypique_iqr,latitude,longitude
0,69002006,84 QUAI PERRACHE,Lyon,69002,2.489,2026-10-14 15:21:57+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,16.663318,True,False,45.73500,4.818000
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,2.429,2026-10-14 20:50:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.195818,True,False,45.74700,4.867000
2,69003013,258 rue Garibaldi,Lyon,69003,2.449,2026-10-05 21:45:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,226.279151,False,False,45.75446,4.852837
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,2.343,2026-10-15 07:32:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,0.495818,True,False,45.77600,4.821000
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,2.469,2026-10-14 21:00:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.029151,True,False,45.77900,4.828000


### C4.7 - Export des données préparées en CSV

Objectif : sauvegarder le DataFrame contrôlé
dans le dossier data/processed/.

Le fichier CSV servira à l'importation dans BigQuery.

Les données brutes JSON sont conservées séparément
dans data/raw/.

In [34]:
# Définir le dossier de destination
dossier_processed = Path("../data/processed")

# Créer le dossier s'il n'existe pas
dossier_processed.mkdir(parents=True, exist_ok=True)

# Construire le nom du CSV à partir de celui du JSON
nom_csv = chemin_fichier.stem + ".csv"

# Construire le chemin complet
chemin_csv = dossier_processed / nom_csv

# Exporter le DataFrame
df_export.to_csv(
    chemin_csv,
    index=False,
    encoding="utf-8"
)

print("CSV enregistré :", chemin_csv.resolve())

CSV enregistré : C:\Users\bowma\optifleet-logistics\data\processed\carburants_69_2026-10-15_080144.csv


In [35]:
# Relire le fichier CSV
df_verification_csv = pd.read_csv(
    chemin_csv,
    dtype={
        "cp": "string",
        "code_departement": "string"
    }
)

# Comparer les dimensions
print("Dimensions avant export :", df_export.shape)
print("Dimensions après relecture :", df_verification_csv.shape)

# Vérifier l'existence du fichier
print("Fichier existe :", chemin_csv.exists())

# Vérifier les identifiants uniques
print("Stations uniques :", df_verification_csv["id"].nunique())

# Afficher les premières lignes
display(df_verification_csv.head())

Dimensions avant export : (200, 14)
Dimensions après relecture : (200, 14)
Fichier existe : True
Stations uniques : 200


,id,adresse,ville,cp,gazole_prix,gazole_maj,code_departement,horodatage_collecte,date_collecte,age_prix_heures,prix_recent_48h,prix_atypique_iqr,latitude,longitude
0,69002006,84 QUAI PERRACHE,Lyon,69002,2.489,2026-10-14 15:21:57+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,16.663318,True,False,45.73500,4.818000
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,2.429,2026-10-14 20:50:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.195818,True,False,45.74700,4.867000
2,69003013,258 rue Garibaldi,Lyon,69003,2.449,2026-10-05 21:45:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,226.279151,False,False,45.75446,4.852837
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,2.343,2026-10-15 07:32:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,0.495818,True,False,45.77600,4.821000
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,2.469,2026-10-14 21:00:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.029151,True,False,45.77900,4.828000


### C4.8 - Premier KPI métier : écart médiane − P10

**Objectif :** calculer l'écart entre le prix médian du Gazole
et le 10e percentile (P10).

Cet écart sert à estimer les économies potentielles
pour OptiFleet Logistics.

**Hypothèse :** consommation mensuelle de 10 400 litres.

**Limite :** le calcul porte encore sur le département 69.
Il sera recalculé sur la Métropole de Lyon après C5.

In [37]:
# C4.8 — Calculer le premier KPI métier

# Conserver uniquement les prix renseignés pour le calcul
prix_valides = df["gazole_prix"].dropna()

# Calculer la médiane et le P10
mediane = prix_valides.median()
p10 = prix_valides.quantile(0.10)

# Calculer l'écart
ecart_prix = mediane - p10

# Estimer les économies mensuelles
volume_mensuel = 10400
economie_mensuelle = ecart_prix * volume_mensuel

# Afficher les résultats
print(f"Médiane : {mediane:.3f} €/L")
print(f"P10 : {p10:.3f} €/L")
print(f"Écart médiane - P10 : {ecart_prix:.3f} €/L")
print(f"Économie mensuelle théorique : {economie_mensuelle:.0f} €")

Médiane : 2.359 €/L
P10 : 2.250 €/L
Écart médiane - P10 : 0.109 €/L
Économie mensuelle théorique : 1134 €


### C4.9 - Test de sensibilité du seuil de fraîcheur

**Objectif :** comparer la couverture des stations
selon trois seuils de fraîcheur :

- 48 heures (2 jours)
- 72 heures (3 jours)
- 168 heures (7 jours)

Une station est éligible si son prix Gazole est renseigné
et si son âge est compris entre 0 et le seuil étudié.

Les dates futures apparentes restent exclues provisoirement.

In [38]:
# C4.9 — Tester différents seuils de fraîcheur

# Compter les stations dont le prix Gazole est renseigné
prix_renseignes = df["gazole_prix"].notna()
total_prix_renseignes = prix_renseignes.sum()

# Tester trois seuils
for seuil in [48, 72, 168]:

    # Sélectionner les prix renseignés et suffisamment récents
    eligibles = (
        prix_renseignes
        & df["age_prix_heures"].between(0, seuil)
    )

    # Compter les stations éligibles
    nombre = eligibles.sum()

    # Calculer leur proportion
    pourcentage = nombre / total_prix_renseignes * 100

    print(
        f"Seuil {seuil} h : "
        f"{nombre} stations ({pourcentage:.1f} %)"
    )

Seuil 48 h : 121 stations (63.0 %)
Seuil 72 h : 133 stations (69.3 %)
Seuil 168 h : 153 stations (79.7 %)


### C4 - Conclusion des analyses métier

**Premier KPI :**

- Prix médian du Gazole : 2,359 €/L.
- P10 : 2,250 €/L.
- Écart médiane − P10 : 0,109 €/L.
- Économie mensuelle théorique : environ 1 134 €,
  sur la base de 10 400 litres par mois.

**Test de sensibilité :**

- 48 h : 121 stations éligibles (63,0 %).
- 72 h : 133 stations éligibles (69,3 %).
- 168 h : 153 stations éligibles (79,7 %).

**Choix provisoire :** retenir 48 h pour privilégier
la fraîcheur des prix et conserver 72 h comme scénario
alternatif.

**Limites :**

- Analyses réalisées sur le département 69.
- Le périmètre Métropole de Lyon reste à appliquer.
- Les anomalies temporelles apparentes restent à examiner.
- Les économies calculées sont théoriques.

**Prochaine étape :** C5 — Filtrage géographique
sur les communes de la Métropole de Lyon.

# C5 - Délimitation géographique : Métropole de Lyon

## Objectif

Identifier les stations-service situées dans la Métropole de Lyon
parmi les stations collectées dans le département 69.

## Méthodologie

1. Obtenir une liste officielle des communes de la Métropole.
2. Comparer les communes des stations avec cette liste.
3. Identifier les stations appartenant au périmètre.
4. Contrôler les communes non reconnues.
5. Créer un DataFrame dédié à l'analyse métier.

**Important :** conserver toutes les stations dans les données brutes.
Le filtrage géographique sera appliqué uniquement aux données d'analyse.

In [36]:
# C5.1 - Explorer les communes présentes dans nos données

# Compter les noms de communes différents
nb_communes = df["ville"].nunique()

print("Nombre de noms de communes distincts :", nb_communes)

# Compter le nombre de stations par commune
stations_par_commune = (
    df["ville"]
    .value_counts()
    .rename_axis("commune")
    .reset_index(name="nombre_stations")
)

# Afficher le tableau complet
display(stations_par_commune)

Nombre de noms de communes distincts : 99


,commune,nombre_stations
0,Lyon,19
1,Saint-Priest,10
2,Villefranche-sur-Saône,8
3,Bron,6
4,Villeurbanne,6
...,...,...
94,Beaujeu,1
95,Lantignié,1
96,Quincié-en-Beaujolais,1
97,Chénelette,1


## C5.2 - Récupération du référentiel officiel

Objectif : récupérer les communes de la Métropole de Lyon
à partir de l'API Géo.

Nous utiliserons les codes INSEE pour fiabiliser
le périmètre géographique.

Les noms de communes seront conservés pour
l'affichage et les contrôles.

In [67]:
# C5.2 — Récupérer les communes de la Métropole de Lyon

# 1. Identifier la Métropole de Lyon
code_metropole = "200046977"

# 2. Construire l'URL de l'API Géo
url_communes = (
    f"https://geo.api.gouv.fr/epcis/{code_metropole}/communes"
)

# 3. Demander les champs nécessaires
parametres = {
    "fields": "nom,code,codesPostaux",
    "format": "json"
}
# 4. Envoyer la requête à l'API Géo
reponse_geo = requests.get(
    url_communes,
    params=parametres,
    timeout=30
)

# 5. Vérifier le statut HTTP
print("Statut HTTP :", reponse_geo.status_code)
reponse_geo.raise_for_status()

# 6. Convertir le JSON en DataFrame
communes_json = reponse_geo.json()
df_communes = pd.DataFrame(communes_json)

# 7. Examiner le résultat
print("Nombre de communes :", len(df_communes))
print("Colonnes disponibles :", df_communes.columns.tolist())

display(df_communes.head())

Statut HTTP : 200
Nombre de communes : 58
Colonnes disponibles : ['nom', 'code', 'codesPostaux']


,nom,code,codesPostaux
0,Albigny-sur-Saône,69003,[69250]
1,Bron,69029,[69500]
2,Cailloux-sur-Fontaines,69033,[69270]
3,Caluire-et-Cuire,69034,[69300]
4,Champagne-au-Mont-d'Or,69040,[69410]


In [40]:
# C5.2 — Vérifier la qualité du référentiel

# Vérifier le nombre de communes
assert len(df_communes) == 58, "Le nombre de communes est inattendu"

# Vérifier les codes INSEE uniques
print("Communes :", len(df_communes))
print("Codes INSEE uniques :", df_communes["code"].nunique())

# Vérifier les valeurs manquantes
print("\nValeurs manquantes :")
print(df_communes.isna().sum())

# Vérifier les doublons de codes INSEE
print(
    "\nDoublons de codes INSEE :",
    df_communes["code"].duplicated().sum()
)

Communes : 58
Codes INSEE uniques : 58

Valeurs manquantes :
nom             0
code            0
codesPostaux    0
dtype: int64

Doublons de codes INSEE : 0


In [41]:
# C5.2 — Sauvegarder le référentiel officiel des communes

import json
from pathlib import Path

# 1. Définir le dossier de sauvegarde
dossier_raw = Path("../data/raw")

# 2. Créer le dossier s'il n'existe pas
dossier_raw.mkdir(parents=True, exist_ok=True)

# 3. Définir le nom du fichier JSON
chemin_communes = dossier_raw / "communes_metropole_lyon.json"

# 4. Enregistrer la réponse brute de l'API
with open(chemin_communes, "w", encoding="utf-8") as fichier:
    json.dump(
        communes_json,
        fichier,
        ensure_ascii=False,
        indent=2
    )

# 5. Vérifier que le fichier existe
print("Fichier créé :", chemin_communes.exists())
print("Chemin :", chemin_communes.resolve())

Fichier créé : True
Chemin : C:\Users\bowma\optifleet-logistics\data\raw\communes_metropole_lyon.json


### C5.3.1 - Préparation des codes postaux avec `.explode()`

**Objectif :** transformer les listes de codes postaux du référentiel officiel en lignes individuelles afin de préparer la correspondance avec les stations-service.

**Précaution :** le code postal ne suffit pas à identifier une commune de manière certaine. Nous conserverons également le nom et le code INSEE pour contrôler les correspondances.

In [43]:
# C5.3.1 — Examiner les codes postaux avant transformation

display(
    df_communes.loc[
        df_communes["nom"].isin(["Lyon", "Bron"]),
        ["nom", "code", "codesPostaux"]
    ]
)

# Vérifier le type de la première valeur
print(
    "Type de codesPostaux :",
    type(df_communes["codesPostaux"].iloc[0])
)

,nom,code,codesPostaux
1,Bron,69029,[69500]
22,Lyon,69123,"[69001, 69002, 69003, 69004, 69005, 69006, 690..."


Type de codesPostaux : <class 'list'>


In [44]:
# C5.3.1 — Déplier les listes de codes postaux

# Créer un nouveau DataFrame sans modifier le référentiel original
df_communes_cp = df_communes.explode(
    "codesPostaux",
    ignore_index=True
)

# Renommer la colonne pour préparer la jointure
df_communes_cp = df_communes_cp.rename(
    columns={"codesPostaux": "cp"}
)

# Vérifier les dimensions
print("Avant explode :", df_communes.shape)
print("Après explode :", df_communes_cp.shape)

# Examiner Lyon et Bron après transformation
display(
    df_communes_cp.loc[
        df_communes_cp["nom"].isin(["Lyon", "Bron"]),
        ["nom", "code", "cp"]
    ]
)

Avant explode : (58, 3)
Après explode : (67, 3)


,nom,code,cp
1,Bron,69029,69500
22,Lyon,69123,69001
23,Lyon,69123,69002
24,Lyon,69123,69003
25,Lyon,69123,69004
26,Lyon,69123,69005
27,Lyon,69123,69006
28,Lyon,69123,69007
29,Lyon,69123,69008
30,Lyon,69123,69009


### C5.3.2 -Contrôle des codes postaux avant la jointure

**Objectif :** vérifier si plusieurs communes du référentiel partagent un même code postal.

**Risque métier :** une jointure uniquement sur le code postal peut attribuer une station à une mauvaise commune ou multiplier les lignes.

Nous contrôlons donc les correspondances avant de filtrer les stations de la Métropole de Lyon.

In [45]:
# C5.3.2 — Vérifier les codes postaux du référentiel

# 1. Compter les lignes et les codes postaux uniques
print("Nombre de lignes :", len(df_communes_cp))
print("Codes postaux uniques :", df_communes_cp["cp"].nunique())

# 2. Repérer les codes postaux présents plusieurs fois
codes_partages = df_communes_cp.loc[
    df_communes_cp["cp"].duplicated(keep=False),
    ["nom", "code", "cp"]
].sort_values("cp")

# 3. Afficher les résultats
print("Lignes avec un code postal partagé :", len(codes_partages))

display(codes_partages)

Nombre de lignes : 67
Codes postaux uniques : 52
Lignes avec un code postal partagé : 22


,nom,code,cp
0,Albigny-sur-Saône,69003,69250
61,Montanay,69284,69250
33,Neuville-sur-Saône,69143,69250
13,Fleurieu-sur-Saône,69085,69250
10,Curis-au-Mont-d'Or,69071,69250
36,Poleymieux-au-Mont-d'Or,69153,69250
8,Couzon-au-Mont-d'Or,69068,69270
14,Fontaines-Saint-Martin,69087,69270
15,Fontaines-sur-Saône,69088,69270
38,Rochetaillée-sur-Saône,69168,69270


### C5.3.3 - Normalisation des noms de communes

**Objectif :** préparer une comparaison fiable entre les noms des villes
de l'API carburants et ceux du référentiel officiel.

Les différences de majuscules, d'espaces et d'accents peuvent empêcher
une jointure de trouver une correspondance.

Nous créons une colonne `ville_normalisee` dans les stations et
`nom_normalise` dans le référentiel, sans modifier les colonnes originales.

La normalisation ne résout pas automatiquement les abréviations
(St / Saint), les changements de noms ou les erreurs de saisie.

In [46]:
# C5.3.3 — Normaliser les noms de communes

# 1. Importer l'outil de gestion des caractères Unicode
import unicodedata

# 2. Définir une fonction de normalisation
def normaliser_commune(nom):

    # Gérer les valeurs manquantes
    if pd.isna(nom):
        return None

    # Supprimer les espaces autour et passer en majuscules
    nom = str(nom).strip().upper()

    # Décomposer les caractères accentués
    nom = unicodedata.normalize("NFD", nom)

    # Supprimer les accents
    nom = "".join(
        caractere
        for caractere in nom
        if unicodedata.category(caractere) != "Mn"
    )

    return nom


# 3. Normaliser les noms des stations
df["ville_normalisee"] = df["ville"].apply(normaliser_commune)

# 4. Normaliser les noms du référentiel
df_communes_cp["nom_normalise"] = (
    df_communes_cp["nom"].apply(normaliser_commune)
)

# 5. Vérifier les résultats
print("STATIONS :")
display(df[["ville", "ville_normalisee"]].head(10))

print("RÉFÉRENTIEL :")
display(
    df_communes_cp[
        ["nom", "nom_normalise", "cp"]
    ].head(10)
)

STATIONS :


,ville,ville_normalisee
0,Lyon,LYON
1,Lyon,LYON
2,Lyon,LYON
3,Lyon,LYON
4,Lyon,LYON
5,Lyon,LYON
6,Lyon,LYON
7,Lyon,LYON
8,Lyon,LYON
9,Lyon,LYON


RÉFÉRENTIEL :


,nom,nom_normalise,cp
0,Albigny-sur-Saône,ALBIGNY-SUR-SAONE,69250
1,Bron,BRON,69500
2,Cailloux-sur-Fontaines,CAILLOUX-SUR-FONTAINES,69270
3,Caluire-et-Cuire,CALUIRE-ET-CUIRE,69300
4,Champagne-au-Mont-d'Or,CHAMPAGNE-AU-MONT-D'OR,69410
5,Charbonnières-les-Bains,CHARBONNIERES-LES-BAINS,69260
6,Charly,CHARLY,69390
7,Collonges-au-Mont-d'Or,COLLONGES-AU-MONT-D'OR,69660
8,Couzon-au-Mont-d'Or,COUZON-AU-MONT-D'OR,69270
9,Craponne,CRAPONNE,69290


### C5.3.4 - Vérification des clés de jointure

**Objectif :** vérifier que chaque couple
`(nom_normalise, cp)` identifie une seule commune
dans le référentiel.

**Pourquoi ?**

Un code postal peut correspondre à plusieurs communes.
En revanche, nous souhaitons que la combinaison
nom normalisé + code postal soit unique.

Cette vérification permet de limiter le risque de
dupliquer les stations lors de la future jointure.

In [47]:
# C5.3.4 — Vérifier l'unicité des clés de jointure

# 1. Définir les deux colonnes utilisées pour la jointure
cles_jointure = ["nom_normalise", "cp"]

# 2. Repérer les lignes dont le couple apparaît plusieurs fois
doublons_cles = df_communes_cp[
    df_communes_cp.duplicated(
        subset=cles_jointure,
        keep=False
    )
]

# 3. Afficher les résultats
print("Lignes du référentiel :", len(df_communes_cp))
print("Clés de jointure uniques :",
      df_communes_cp[cles_jointure].drop_duplicates().shape[0])

print("Lignes avec une clé dupliquée :", len(doublons_cles))

display(doublons_cles)

Lignes du référentiel : 67
Clés de jointure uniques : 67
Lignes avec une clé dupliquée : 0


,nom,code,cp,nom_normalise


### C5.3.5 - Première jointure géographique contrôlée

**Objectif :** rapprocher les stations du Rhône avec le
référentiel officiel des communes de la Métropole de Lyon.

**Clés de jointure :**
- Nom de commune normalisé
- Code postal

**Méthode :** jointure gauche (`left join`) avec `pd.merge()`.

**Contrôles :**
- `validate="many_to_one"` : plusieurs stations peuvent
  appartenir à une commune, mais une clé du référentiel
  ne doit correspondre qu'à une seule ligne.
- `indicator=True` : identifier les stations ayant trouvé
  une correspondance et celles sans correspondance.

Les stations non correspondantes seront examinées avant
toute exclusion définitive.

In [48]:
# C5.3.5 — Première jointure géographique contrôlée

# 1. Préparer les codes postaux comme chaînes de caractères
df["cp"] = df["cp"].astype("string").str.strip()

df_communes_cp["cp"] = (
    df_communes_cp["cp"].astype("string").str.strip()
)

# 2. Réaliser une jointure gauche
df_jointure = pd.merge(
    left=df,
    right=df_communes_cp[
        ["nom", "code", "nom_normalise", "cp"]
    ],
    how="left",
    left_on=["ville_normalisee", "cp"],
    right_on=["nom_normalise", "cp"],
    validate="many_to_one",
    indicator=True
)

# 3. Vérifier les dimensions
print("Stations avant jointure :", len(df))
print("Lignes après jointure :", len(df_jointure))

# 4. Compter les correspondances
print("\nRésultat de la jointure :")
print(df_jointure["_merge"].value_counts(dropna=False))

# 5. Afficher quelques lignes
display(
    df_jointure[
        [
            "id",
            "ville",
            "cp",
            "nom",
            "code",
            "_merge"
        ]
    ].head(10)
)

Stations avant jointure : 200
Lignes après jointure : 200

Résultat de la jointure :
_merge
both          108
left_only      92
right_only      0
Name: count, dtype: int64


,id,ville,cp,nom,code,_merge
0,69002006,Lyon,69002,Lyon,69123,both
1,69003012,Lyon,69003,Lyon,69123,both
2,69003013,Lyon,69003,Lyon,69123,both
3,69004002,Lyon,69004,Lyon,69123,both
4,69004016,Lyon,69004,Lyon,69123,both
5,69004020,Lyon,69004,Lyon,69123,both
6,69005002,Lyon,69005,Lyon,69123,both
7,69005005,Lyon,69005,Lyon,69123,both
8,69007006,Lyon,69007,Lyon,69123,both
9,69007008,Lyon,69007,Lyon,69123,both


### C5.3.6 - Analyse des stations sans correspondance

**Objectif :** examiner les stations du Rhône qui n'ont pas
trouvé de correspondance dans le référentiel de la Métropole.

Une absence de correspondance peut signifier :

- Une station située hors de la Métropole de Lyon.
- Une différence d'orthographe ou de nom de commune.
- Un code postal différent ou incohérent.
- Une anomalie dans les données sources.

**Méthode :** isoler les stations `left_only`, puis analyser
leurs noms de communes et leurs codes postaux.

Aucune station ne sera supprimée à cette étape.

In [49]:
# C5.3.6 — Examiner les stations sans correspondance

# 1. Isoler les stations sans correspondance
df_non_correspondantes = df_jointure.loc[
    df_jointure["_merge"] == "left_only"
].copy()

# 2. Vérifier le nombre de stations concernées
print(
    "Stations sans correspondance :",
    len(df_non_correspondantes)
)

# 3. Compter les stations par commune
communes_non_correspondantes = (
    df_non_correspondantes
    .groupby(["ville", "cp"], dropna=False)
    .agg(nombre_stations=("id", "count"))
    .reset_index()
    .sort_values(
        "nombre_stations",
        ascending=False
    )
)

# 4. Afficher toutes les communes concernées
display(communes_non_correspondantes)

Stations sans correspondance : 92


,ville,cp,nombre_stations
58,Villefranche-sur-Saône,69400,7
5,Belleville-en-Beaujolais,69220,5
0,Amplepuis,69550,3
2,Anse,69480,3
60,Vindry-sur-Turdine,69490,3
...,...,...,...
29,Loire-sur-Rhône,69700,1
30,Lozanne,69380,1
1,Ampuis,69420,1
32,Lyon,69200,1


### C5.3.7 - Recherche des correspondances géographiques potentielles

**Objectif :** identifier, parmi les stations sans correspondance
exacte, celles dont le code postal existe dans le référentiel
de la Métropole de Lyon.

**Hypothèse :** certaines stations peuvent appartenir à la
Métropole malgré une différence entre le nom de ville
déclaré et le nom officiel.

**Attention :** un code postal partagé ne suffit pas
à prouver l'appartenance à la Métropole.

Les résultats seront considérés comme des anomalies
ou des correspondances potentielles à vérifier.

In [51]:
# C5.3.7 — Rechercher les codes postaux de la Métropole

# 1. Construire l'ensemble des codes postaux officiels
cp_metropole = set(df_communes_cp["cp"].dropna())

# 2. Vérifier si le code postal des stations non reconnues
#    existe dans le référentiel
df_non_correspondantes["cp_dans_metropole"] = (
    df_non_correspondantes["cp"].isin(cp_metropole)
)

# 3. Compter les stations candidates à une vérification
print("Stations non reconnues :", len(df_non_correspondantes))

print(
    "Stations avec un CP présent dans le référentiel :",
    df_non_correspondantes["cp_dans_metropole"].sum()
)

print(
    "Stations avec un CP absent du référentiel :",
    (~df_non_correspondantes["cp_dans_metropole"]).sum()
)

# 4. Afficher les stations à examiner
stations_a_verifier = df_non_correspondantes.loc[
    df_non_correspondantes["cp_dans_metropole"],
    ["id", "ville", "cp", "adresse", "geom"]
]

display(stations_a_verifier)

Stations non reconnues : 92
Stations avec un CP présent dans le référentiel : 22
Stations avec un CP absent du référentiel : 70


,id,ville,cp,adresse,geom
18,69100022,Bron,69100,283 RTE DE GENAS,"{'lon': 4.91257, 'lat': 45.74977}"
24,69110004,Lyon,69110,Rue Commandant Charcot,"{'lon': 4.794, 'lat': 45.749}"
49,69200003,Lyon,69200,38 rue du moulin à vent,"{'lon': 4.85621, 'lat': 45.73012}"
79,69290002,Grézieu-la-Varenne,69290,Avenue Benoît Launay,"{'lon': 4.692, 'lat': 45.741}"
91,69330006,Pusignan,69330,15 ROUTE DE JONS,"{'lon': 5.0757425322646, 'lat': 45.76528243325..."
95,69360010,Sérézin-du-Rhône,69360,2 AVENUE DU DAUPHINE,"{'lon': 4.825, 'lat': 45.629}"
97,69360012,Communay,69360,AUTOROUTE A 46 - AIRE COMMUNAY NORD,"{'lon': 4.826, 'lat': 45.591}"
98,69360013,Communay,69360,AUTOROUTE A46 - AIRE COMMUNAY SUD,"{'lon': 4.826, 'lat': 45.589}"
99,69360015,Saint-Symphorien-d'Ozon,69360,ROUTE DE CORBAS CD 148,"{'lon': 4.869, 'lat': 45.636}"
100,69360018,Sérézin-du-Rhône,69360,A7 AIRE DE SEREZIN DU RHONE,"{'lon': 4.819, 'lat': 45.631}"


### C5.3.8 - Examiner les codes postaux partagés

**Objectif :** comprendre pourquoi certaines stations n'ont pas été
reconnues malgré un code postal présent dans le référentiel.

Nous comparons les communes déclarées par les stations avec
les communes officielles associées au même code postal.

**Précaution :** cette analyse sert à identifier des anomalies.
Elle ne constitue pas une validation géographique définitive.

In [54]:
# C5.3.8 — Comparer les villes déclarées et les communes officielles

# 1. Récupérer les 22 stations candidates
df_candidates = df_non_correspondantes.loc[
    df_non_correspondantes["cp_dans_metropole"]
].copy()

# 2. # Retrouver les communes officielles partageant le code postal
df_comparaison_cp = df_candidates.merge(
    df_communes_cp[["cp", "nom", "code"]],
    on="cp",
    how="left",
    validate="many_to_many",
    suffixes=("_ancienne_jointure", "_referentiel")
)

# 3. # Afficher les correspondances possibles
display(
    df_comparaison_cp[
        [
            "id",
            "ville",
            "cp",
            "nom_referentiel",
            "code_referentiel"
        ]
    ].sort_values(["cp", "ville"])
)

# 4. Contrôler le nombre de stations distinctes
print("Stations candidates :", df_candidates["id"].nunique())
print("Lignes après jointure :", len(df_comparaison_cp))
print(
    "Stations distinctes après jointure :",
    df_comparaison_cp["id"].nunique()
)

,id,ville,cp,nom_referentiel,code_referentiel
0,69100022,Bron,69100,Villeurbanne,69266
1,69110004,Lyon,69110,Sainte-Foy-lès-Lyon,69202
2,69200003,Lyon,69200,Vénissieux,69259
3,69290002,Grézieu-la-Varenne,69290,Craponne,69069
4,69290002,Grézieu-la-Varenne,69290,Saint-Genis-les-Ollières,69205
5,69330006,Pusignan,69330,Jonage,69279
6,69330006,Pusignan,69330,Meyzieu,69282
8,69360012,Communay,69360,Solaize,69296
9,69360013,Communay,69360,Solaize,69296
10,69360015,Saint-Symphorien-d'Ozon,69360,Solaize,69296


Stations candidates : 22
Lignes après jointure : 26
Stations distinctes après jointure : 22


In [55]:
# C5.3.9 — Vérifier les effets de la jointure par code postal

print("Stations candidates :", df_candidates["id"].nunique())

print("Lignes après jointure :", len(df_comparaison_cp))

print(
    "Stations distinctes après jointure :",
    df_comparaison_cp["id"].nunique()
)

print(
    "Nombre de codes postaux partagés parmi les candidats :",
    df_comparaison_cp.groupby("id").size().gt(1).sum()
)

Stations candidates : 22
Lignes après jointure : 26
Stations distinctes après jointure : 22
Nombre de codes postaux partagés parmi les candidats : 4


### C5.3.10 - Vérification des coordonnées géographiques

**Objectif :** vérifier que les stations possèdent des coordonnées
GPS valides avant de les comparer aux frontières de la Métropole.

Une station est représentée par :
- sa latitude : position nord-sud ;
- sa longitude : position est-ouest.

**Attention :** avoir des coordonnées GPS valides ne signifie
pas encore appartenir à la Métropole de Lyon.

Cette étape prépare un contrôle géographique plus fiable
que la seule comparaison des noms et des codes postaux.

In [57]:
# C5.3.10 — Diagnostic des coordonnées géographiques

# Vérifier les colonnes disponibles
print("Colonnes de df :")
print(df.columns.tolist())

# Vérifier la présence de geom
print("\nExemple de coordonnées brutes :")
display(df[["id", "ville", "geom"]].head())

# Vérifier les dimensions
print("\nDimensions :", df.shape)

Colonnes de df :
['id', 'adresse', 'ville', 'cp', 'geom', 'gazole_prix', 'gazole_maj', 'code_departement', 'horodatage_collecte', 'date_collecte', 'age_prix_heures', 'prix_recent_48h', 'prix_atypique_iqr', 'ville_normalisee']

Exemple de coordonnées brutes :


,id,ville,geom
0,69002006,Lyon,"{'lon': 4.818, 'lat': 45.735}"
1,69003012,Lyon,"{'lon': 4.867, 'lat': 45.747}"
2,69003013,Lyon,"{'lon': 4.8528365475131, 'lat': 45.7544602}"
3,69004002,Lyon,"{'lon': 4.821, 'lat': 45.776}"
4,69004016,Lyon,"{'lon': 4.828, 'lat': 45.779}"



Dimensions : (200, 14)


### C5.3.11 - Extraction des coordonnées GPS

**Objectif :** transformer la colonne `geom` en deux colonnes numériques,
`latitude` et `longitude`, afin de préparer le contrôle géographique
des stations de la Métropole de Lyon.

Nous conservons la colonne `geom` d'origine pour assurer la traçabilité.

In [58]:
# C5.3.11 — Extraire les coordonnées GPS depuis geom

# 1. Examiner le type des valeurs de geom
print("Types de données dans geom :")
print(df["geom"].apply(type).value_counts())

# 2. Extraire la latitude
df["latitude"] = df["geom"].apply(
    lambda coord: coord.get("lat") if isinstance(coord, dict) else None
)

# 3. Extraire la longitude
df["longitude"] = df["geom"].apply(
    lambda coord: coord.get("lon") if isinstance(coord, dict) else None
)

# 4. Convertir les coordonnées en valeurs numériques
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# 5. Vérifier le résultat
print("\nDimensions après extraction :", df.shape)

display(
    df[["id", "ville", "geom", "latitude", "longitude"]].head()
)

print("\nCoordonnées manquantes :")
print(df[["latitude", "longitude"]].isna().sum())

Types de données dans geom :
geom
<class 'dict'>    200
Name: count, dtype: int64

Dimensions après extraction : (200, 16)


,id,ville,geom,latitude,longitude
0,69002006,Lyon,"{'lon': 4.818, 'lat': 45.735}",45.73500,4.818000
1,69003012,Lyon,"{'lon': 4.867, 'lat': 45.747}",45.74700,4.867000
2,69003013,Lyon,"{'lon': 4.8528365475131, 'lat': 45.7544602}",45.75446,4.852837
3,69004002,Lyon,"{'lon': 4.821, 'lat': 45.776}",45.77600,4.821000
4,69004016,Lyon,"{'lon': 4.828, 'lat': 45.779}",45.77900,4.828000



Coordonnées manquantes :
latitude     0
longitude    0
dtype: int64


### C5.3.12 - Contrôle de validité des coordonnées GPS

**Objectif :** vérifier que les coordonnées GPS des 200 stations
respectent les limites géographiques possibles.

Nous contrôlerons :
- Les coordonnées manquantes.
- Les latitudes comprises entre -90 et 90.
- Les longitudes comprises entre -180 et 180.
- Les coordonnées égales à (0, 0), suspectes pour une station du Rhône.

Ce contrôle ne suffit pas encore à confirmer qu'une station
appartient à la Métropole de Lyon.

In [59]:
# C5.3.12 — Contrôler la validité des coordonnées GPS

# 1. Vérifier les limites possibles des latitudes
latitude_valide = df["latitude"].between(-90, 90)

# 2. Vérifier les limites possibles des longitudes
longitude_valide = df["longitude"].between(-180, 180)

# 3. Repérer les coordonnées (0, 0)
coordonnees_zero = (
    (df["latitude"] == 0)
    & (df["longitude"] == 0)
)

# 4. Repérer les stations avec des coordonnées problématiques
coordonnees_invalides = (
    ~latitude_valide
    | ~longitude_valide
    | coordonnees_zero
)

# 5. Afficher le bilan
print("Nombre total de stations :", len(df))
print("Latitudes hors limites ou manquantes :", (~latitude_valide).sum())
print("Longitudes hors limites ou manquantes :", (~longitude_valide).sum())
print("Coordonnées (0, 0) :", coordonnees_zero.sum())
print("Stations avec coordonnées problématiques :", coordonnees_invalides.sum())

# 6. Afficher les stations à examiner
display(
    df.loc[
        coordonnees_invalides,
        ["id", "ville", "latitude", "longitude"]
    ]
)

Nombre total de stations : 200
Latitudes hors limites ou manquantes : 0
Longitudes hors limites ou manquantes : 0
Coordonnées (0, 0) : 0
Stations avec coordonnées problématiques : 0


,id,ville,latitude,longitude


### C5.3.13 - Préparation du contrôle d'appartenance géographique

**Objectif :** déterminer quelles stations appartiennent réellement
à la Métropole de Lyon.

**Méthode envisagée :**
1. Récupérer les contours géographiques officiels des 58 communes.
2. Associer chaque station à un point GPS.
3. Vérifier si ce point se trouve dans une commune du référentiel.
4. Comparer ce résultat à la première jointure nom + code postal.

**Précaution :** ne pas exclure automatiquement les 92 stations
non reconnues par la première jointure.

### C5.3.14 - Récupération des contours géographiques officiels

**Objectif :** récupérer les géométries des communes
de la Métropole de Lyon à partir de l'API Géo.

**Principe :**
- Réutiliser le code de l'EPCI de la Métropole.
- Demander les informations géographiques des communes.
- Vérifier le nombre de communes et le format des géométries.
- Conserver le référentiel original sans le modifier.

**Attention :** une géométrie valide ne garantit pas
à elle seule que les coordonnées d'une station sont correctes.

In [60]:
# C5.3.14 — Préparer la récupération des contours

# 1. Adresse de l'API Géo
url_contours = "https://geo.api.gouv.fr/epcis/200046977/communes"

# 2. Paramètres demandés à l'API
parametres_contours = {
    "fields": "nom,code,contour",
    "format": "geojson",
    "geometry": "contour"
}

# 3. Examiner notre requête avant de l'envoyer
print("URL de l'API :", url_contours)
print("Paramètres :", parametres_contours)

URL de l'API : https://geo.api.gouv.fr/epcis/200046977/communes
Paramètres : {'fields': 'nom,code,contour', 'format': 'geojson', 'geometry': 'contour'}


In [62]:
# C5.3.14 — Envoyer la requête à l'API Géo

# 1. Envoyer la requête HTTP
reponse_contours = requests.get(
    url_contours,
    params=parametres_contours,
    timeout=30
)

# 2. Vérifier la réponse du serveur
print("Statut HTTP :", reponse_contours.status_code)

# 3. Afficher l'URL réellement envoyée
print("URL complète :", reponse_contours.url)

# 4. Vérifier si le serveur signale une erreur
reponse_contours.raise_for_status()

Statut HTTP : 200
URL complète : https://geo.api.gouv.fr/epcis/200046977/communes?fields=nom%2Ccode%2Ccontour&format=geojson&geometry=contour


In [63]:
# C5.3.14 — Examiner la structure du GeoJSON

# 1. Convertir la réponse HTTP en objet Python
contours_json = reponse_contours.json()

# 2. Vérifier le type Python obtenu
print("Type Python :", type(contours_json))

# 3. Examiner la structure générale
if isinstance(contours_json, dict):
    print("Clés disponibles :", list(contours_json.keys()))
    print("Type GeoJSON :", contours_json.get("type"))

    # 4. Vérifier le nombre d'objets géographiques
    features = contours_json.get("features", [])
    print("Nombre de features :", len(features))

    # 5. Examiner la première géométrie
    if features:
        premiere_feature = features[0]

        print("\nPropriétés de la première commune :")
        print(premiere_feature.get("properties"))

        print("\nType de géométrie :")
        geometrie = premiere_feature.get("geometry") or {}
        print(geometrie.get("type"))

else:
    print("La réponse est une liste de", len(contours_json), "éléments.")
    print("Clés du premier élément :", list(contours_json[0].keys()) if contours_json else [])

Type Python : <class 'dict'>
Clés disponibles : ['type', 'features']
Type GeoJSON : FeatureCollection
Nombre de features : 58

Propriétés de la première commune :
{'nom': 'Albigny-sur-Saône', 'code': '69003'}

Type de géométrie :
Polygon


### C5.3.15 — Sauvegarde du référentiel géographique officiel

**Objectif :** conserver les contours GeoJSON des 58 communes de la Métropole de Lyon dans `data/raw/`.

Le fichier sera conservé sans transformation afin de garantir la traçabilité de la source.

Cette sauvegarde servira ensuite au contrôle spatial des stations-service.

In [65]:
# C5.3.15 — Sauvegarder les contours GeoJSON

# 1. Définir le dossier de sauvegarde
dossier_raw = Path("../data/raw")

# 2. Créer le dossier s'il n'existe pas
dossier_raw.mkdir(parents=True, exist_ok=True)

# 3. Construire le chemin du fichier
chemin_geojson = dossier_raw / "communes_metropole_lyon.geojson"

# 4. Enregistrer la réponse GeoJSON
with open(chemin_geojson, "w", encoding="utf-8") as fichier:
    json.dump(
        contours_json,
        fichier,
        ensure_ascii=False,
        indent=2
    )

# 5. Vérifier la sauvegarde
print("Fichier enregistré :", chemin_geojson.resolve())
print("Fichier existant :", chemin_geojson.exists())
print("Taille du fichier (Ko) :", round(chemin_geojson.stat().st_size / 1024, 1))

Fichier enregistré : C:\Users\bowma\optifleet-logistics\data\raw\communes_metropole_lyon.geojson
Fichier existant : True
Taille du fichier (Ko) : 1117.5


### C5.3.16 - Vérification du fichier GeoJSON sauvegardé

**Objectif :** relire le fichier GeoJSON depuis le disque et vérifier son intégrité.

Contrôles :
- Le fichier contient une collection géographique (`FeatureCollection`).
- Il contient 58 communes.
- Chaque commune possède un code INSEE et une géométrie.
- Identifier les types de géométries présents.

Cette étape prépare la future jointure spatiale entre les stations-service et les communes.

In [66]:
# C5.3.16 — Vérifier le fichier GeoJSON sauvegardé

# 1. Relire le fichier depuis le disque
with open(chemin_geojson, "r", encoding="utf-8") as fichier:
    geojson_verification = json.load(fichier)

# 2. Vérifier le type de document
print("Type GeoJSON :", geojson_verification.get("type"))

# 3. Récupérer les communes
communes_geo = geojson_verification.get("features", [])

print("Nombre de communes :", len(communes_geo))

# 4. Vérifier le nombre attendu
assert len(communes_geo) == 58, "Nombre de communes inattendu"

# 5. Examiner les types de géométries
types_geometries = [
    commune["geometry"]["type"]
    for commune in communes_geo
]

print("Types de géométries :", set(types_geometries))

# 6. Vérifier les informations indispensables
nb_sans_code = sum(
    not commune.get("properties", {}).get("code")
    for commune in communes_geo
)

nb_sans_geometrie = sum(
    commune.get("geometry") is None
    for commune in communes_geo
)

print("Communes sans code INSEE :", nb_sans_code)
print("Communes sans géométrie :", nb_sans_geometrie)

Type GeoJSON : FeatureCollection
Nombre de communes : 58
Types de géométries : {'Polygon'}
Communes sans code INSEE : 0
Communes sans géométrie : 0


### C5.4.1 - Conversion des contours GeoJSON en polygones Shapely

**Objectif :** transformer les 58 géométries officielles de la Métropole de Lyon en objets Shapely pour préparer le contrôle spatial des stations-service.

Nous conservons les codes INSEE et les noms des communes afin de pouvoir identifier la commune associée à chaque station.

In [73]:
%pip install shapely

Note: you may need to restart the kernel to use updated packages.


In [76]:
# C5.4.1 — Transformer les contours GeoJSON en objets Shapely

# 1. Importer la fonction de conversion
from shapely.geometry import shape

# 2. Transformer la géométrie de la première commune
premiere_commune = communes_geo[0]

polygone_test = shape(premiere_commune["geometry"])

# 3. Vérifier le résultat
print("Commune :", premiere_commune["properties"]["nom"])
print("Type de géométrie :", polygone_test.geom_type)
print("Géométrie valide :", polygone_test.is_valid)

Commune : Albigny-sur-Saône
Type de géométrie : Polygon
Géométrie valide : True


### C5.4.2 - Conversion des 58 contours communaux

**Objectif :** convertir toutes les géométries GeoJSON en polygones Shapely, tout en conservant le code INSEE et le nom de chaque commune.

Cette transformation prépare la recherche de la commune géographique de chaque station-service.

In [77]:
# C5.4.2 — Transformer les 58 contours communaux

# 1. Créer une liste vide pour stocker les résultats
polygones_communes = []

# 2. Parcourir chaque commune du référentiel GeoJSON
for commune in communes_geo:

    # Récupérer les informations administratives
    code_insee = commune["properties"]["code"]
    nom_commune = commune["properties"]["nom"]

    # Convertir le contour GeoJSON en géométrie Shapely
    polygone = shape(commune["geometry"])

    # Ajouter les informations dans notre liste
    polygones_communes.append({
        "code_insee": code_insee,
        "nom_commune": nom_commune,
        "geometrie": polygone
    })

# 3. Vérifier le nombre de polygones
print("Nombre de polygones :", len(polygones_communes))

# 4. Vérifier leur validité
nb_valides = sum(
    element["geometrie"].is_valid
    for element in polygones_communes
)

print("Géométries valides :", nb_valides)
print("Géométries invalides :", len(polygones_communes) - nb_valides)

# 5. Examiner la première commune
print("Premier élément :", {
    "code_insee": polygones_communes[0]["code_insee"],
    "nom_commune": polygones_communes[0]["nom_commune"],
    "type_geometrie": polygones_communes[0]["geometrie"].geom_type
})

Nombre de polygones : 58
Géométries valides : 58
Géométries invalides : 0
Premier élément : {'code_insee': '69003', 'nom_commune': 'Albigny-sur-Saône', 'type_geometrie': 'Polygon'}


### C5.4.3 - Transformer les coordonnées GPS d'une station en point Shapely

**Objectif :** créer un objet géographique `Point` à partir de la longitude et de la latitude d'une station-service.

Cette étape prépare la comparaison entre la position GPS des stations et les contours officiels des communes de la Métropole de Lyon.

**Attention :** Shapely attend les coordonnées dans l'ordre longitude, latitude.

In [78]:
# C5.4.3 — Créer un point GPS Shapely

# 1. Importer Point
from shapely.geometry import Point

# 2. Sélectionner la première station
station_test = df.iloc[0]

# 3. Récupérer ses coordonnées GPS
longitude = station_test["longitude"]
latitude = station_test["latitude"]

# 4. Créer le point géographique
point_station = Point(longitude, latitude)

# 5. Vérifier le résultat
print("Identifiant :", station_test["id"])
print("Ville déclarée :", station_test["ville"])
print("Longitude :", longitude)
print("Latitude :", latitude)
print("Type de géométrie :", point_station.geom_type)
print("Point Shapely :", point_station)
print("Point vide :", point_station.is_empty)

Identifiant : 69002006
Ville déclarée : Lyon
Longitude : 4.818
Latitude : 45.735
Type de géométrie : Point
Point Shapely : POINT (4.818 45.735)
Point vide : False


### C5.4.4 - Premier test d'appartenance géographique avec `.covers()`

**Objectif :** vérifier si le point GPS de notre première station-service appartient au contour d'une commune de la Métropole de Lyon.

La méthode `covers()` retourne `True` lorsque le point se trouve à l'intérieur du polygone ou sur sa frontière.

Nous commençons par un seul polygone pour comprendre le fonctionnement avant de rechercher la commune parmi les 58 contours.

In [80]:
# C5.4 — Identifier les stations de la Métropole de Lyon

from shapely.geometry import Point, shape

# 1. Préparer les 58 polygones officiels
polygones = [
    (
        commune["properties"]["code"],
        commune["properties"]["nom"],
        shape(commune["geometry"])
    )
    for commune in communes_geo
]

# 2. Trouver la commune correspondant aux coordonnées GPS
def trouver_commune(longitude, latitude):

    point = Point(longitude, latitude)

    for code, nom, polygone in polygones:
        if polygone.covers(point):
            return code, nom

    return None, None


# 3. Appliquer la recherche aux 200 stations
resultats = df.apply(
    lambda ligne: trouver_commune(
        ligne["longitude"],
        ligne["latitude"]
    ),
    axis=1
)

# 4. Ajouter les résultats au DataFrame
df["code_insee_geo"] = resultats.str[0]
df["commune_geo"] = resultats.str[1]

df["dans_metropole_geo"] = df["code_insee_geo"].notna()

# 5. Créer le DataFrame des stations de la Métropole
df_metropole = df.loc[df["dans_metropole_geo"]].copy()

# 6. Afficher le bilan
print("Stations du Rhône :", len(df))
print("Stations dans la Métropole :", len(df_metropole))
print("Stations hors Métropole :", (~df["dans_metropole_geo"]).sum())

display(
    df_metropole[
        ["id", "ville", "cp", "commune_geo", "gazole_prix"]
    ].head(10)
)

Stations du Rhône : 200
Stations dans la Métropole : 111
Stations hors Métropole : 89


,id,ville,cp,commune_geo,gazole_prix
0,69002006,Lyon,69002,Lyon,2.489
1,69003012,Lyon,69003,Lyon,2.429
2,69003013,Lyon,69003,Lyon,2.449
3,69004002,Lyon,69004,Lyon,2.343
4,69004016,Lyon,69004,Lyon,2.469
5,69004020,Lyon,69004,Lyon,2.469
6,69005002,Lyon,69005,Lyon,2.429
7,69005005,Lyon,69005,Lyon,2.459
8,69007006,Lyon,69007,Lyon,2.250
9,69007008,Lyon,69007,Lyon,2.250


## C5.5 - Préparation des stations de la Métropole

**Objectif :** conserver les stations situées dans la Métropole
de Lyon et disposant d'un prix Gazole renseigné.

La jointure spatiale a identifié 111 stations sur les 200
stations du département du Rhône.

Nous allons maintenant préparer les données utilisées
pour calculer les indicateurs métier d'OptiFleet.

In [81]:
# C5.5 — Préparer les stations avec un prix Gazole exploitable

# 1. Partir uniquement des stations de la Métropole
df_gazole_metropole = df_metropole.copy()

# 2. Convertir le prix Gazole en valeur numérique
df_gazole_metropole["gazole_prix"] = pd.to_numeric(
    df_gazole_metropole["gazole_prix"],
    errors="coerce"
)

# 3. Conserver les prix renseignés et strictement positifs
df_gazole_metropole = df_gazole_metropole.loc[
    df_gazole_metropole["gazole_prix"].notna()
    & (df_gazole_metropole["gazole_prix"] > 0)
].copy()

# 4. Afficher le bilan
print("Stations dans la Métropole :", len(df_metropole))
print("Stations avec prix Gazole exploitable :", len(df_gazole_metropole))
print("Stations sans prix exploitable :",
      len(df_metropole) - len(df_gazole_metropole))

# 5. Aperçu
display(
    df_gazole_metropole[
        ["id", "commune_geo", "gazole_prix", "gazole_maj"]
    ].head(10)
)

Stations dans la Métropole : 111
Stations avec prix Gazole exploitable : 108
Stations sans prix exploitable : 3


,id,commune_geo,gazole_prix,gazole_maj
0,69002006,Lyon,2.489,2026-10-14 15:21:57+00:00
1,69003012,Lyon,2.429,2026-10-14 20:50:00+00:00
2,69003013,Lyon,2.449,2026-10-05 21:45:00+00:00
3,69004002,Lyon,2.343,2026-10-15 07:32:00+00:00
4,69004016,Lyon,2.469,2026-10-14 21:00:00+00:00
5,69004020,Lyon,2.469,2026-10-14 21:00:00+00:00
6,69005002,Lyon,2.429,2026-10-15 08:00:00+00:00
7,69005005,Lyon,2.459,2026-10-05 21:00:00+00:00
8,69007006,Lyon,2.250,2026-10-15 00:01:00+00:00
9,69007008,Lyon,2.250,2026-10-15 00:01:00+00:00


## C5.6 - Recalcul des KPI sur la Métropole de Lyon

**Objectif :** recalculer les indicateurs de prix Gazole sur
le périmètre géographique réel d'OptiFleet Logistics.

Les calculs portent sur les stations de la Métropole disposant
d'un prix Gazole renseigné et strictement positif.

**Attention :** cette première estimation ne filtre pas encore
les prix selon leur fraîcheur.

In [82]:
# C5.6 — KPI Gazole sur la Métropole de Lyon

# 1. Calculer le prix médian
mediane_metropole = df_gazole_metropole["gazole_prix"].median()

# 2. Calculer le 10e percentile (P10)
p10_metropole = df_gazole_metropole["gazole_prix"].quantile(0.10)

# 3. Calculer l'écart de prix par litre
ecart_metropole = mediane_metropole - p10_metropole

# 4. Calculer le volume mensuel théorique d'OptiFleet
volume_mensuel = 20 * 60 * 2 * 52 / 12

# 5. Estimer l'économie mensuelle potentielle
economie_mensuelle = ecart_metropole * volume_mensuel

# 6. Afficher les résultats
print(f"Stations analysées : {len(df_gazole_metropole)}")
print(f"Prix médian : {mediane_metropole:.3f} €/L")
print(f"P10 : {p10_metropole:.3f} €/L")
print(f"Écart médiane - P10 : {ecart_metropole:.3f} €/L")
print(f"Volume mensuel théorique : {volume_mensuel:,.0f} L")
print(f"Économie mensuelle potentielle : {economie_mensuelle:,.0f} €")

Stations analysées : 108
Prix médian : 2.350 €/L
P10 : 2.250 €/L
Écart médiane - P10 : 0.100 €/L
Volume mensuel théorique : 10,400 L
Économie mensuelle potentielle : 1,040 €


In [83]:
# C5.7 — KPI Métropole avec prix récents (< 48 h)

# 1. Sélectionner les stations dont le prix est récent
df_metropole_48h = df_gazole_metropole.loc[
    df_gazole_metropole["age_prix_heures"].between(
        0, 48, inclusive="left"
    )
].copy()

# 2. Calculer les KPI
mediane_48h = df_metropole_48h["gazole_prix"].median()
p10_48h = df_metropole_48h["gazole_prix"].quantile(0.10)

ecart_48h = mediane_48h - p10_48h
economie_48h = ecart_48h * volume_mensuel

# 3. Calculer la couverture des prix récents
couverture_48h = len(df_metropole_48h) / len(df_gazole_metropole) * 100

# 4. Afficher les résultats
print(f"Stations avec prix exploitable : {len(df_gazole_metropole)}")
print(f"Stations avec prix récent (< 48 h) : {len(df_metropole_48h)}")
print(f"Couverture des prix récents : {couverture_48h:.1f} %")
print(f"Médiane récente : {mediane_48h:.3f} €/L")
print(f"P10 récent : {p10_48h:.3f} €/L")
print(f"Écart récent : {ecart_48h:.3f} €/L")
print(f"Économie mensuelle théorique : {economie_48h:,.0f} €")

Stations avec prix exploitable : 108
Stations avec prix récent (< 48 h) : 80
Couverture des prix récents : 74.1 %
Médiane récente : 2.350 €/L
P10 récent : 2.250 €/L
Écart récent : 0.100 €/L
Économie mensuelle théorique : 1,040 €
